# MSc Practical: Object Detection in Images with Ultralytics YOLO — Student Version

Duration: **90 minutes**

## Learning outcomes

By the end of this practical, you should be able to:

1. Explain the input/output structure of a YOLO object detector.
2. Load a pretrained Ultralytics YOLO model.
3. Run object detection on one or more images.
4. Interpret bounding boxes, classes, confidence scores, and result tensors.
5. Visualise detections and apply simple confidence/class filtering.
6. Compare the speed/accuracy trade-off of two model sizes.
7. Reflect on limitations, bias, false positives, false negatives, and deployment constraints.

## How to use this notebook

Some setup and data-loading code is provided for you. The remaining sections contain **TODO** instructions and incomplete code. Complete each task by replacing `...` with your own code.

## Suggested 90-minute structure

| Time | Activity |
|---:|---|
| 0–10 min | Concept recap: object detection, bounding boxes, confidence, classes |
| 10–20 min | Environment setup and first inference |
| 20–35 min | Inspect YOLO result objects |
| 35–50 min | Visualise and filter detections |
| 50–65 min | Batch inference on multiple images |
| 65–80 min | Compare model sizes |
| 80–90 min | Discussion and reflection |


In [ ]:
# Optional: install dependencies.
# python -m pip install --upgrade pip --trusted-host pypi.org --trusted-host files.pythonhosted.org
# python -m pip install ipykernel ultralytics opencv-python matplotlib pandas --trusted-host pypi.org --trusted-host files.pythonhosted.org
# python -m ipykernel install --user --name teste --display-name "teste (Python 3.10.6)"
# In Google Colab, uncomment and run:
#!pip install -q ultralytics opencv-python matplotlib pandas

import sys
print("Python version:", sys.version)

In [ ]:
from pathlib import Path
import time
import urllib.request

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from ultralytics import YOLO
%matplotlib inline

In [ ]:
# TODO: Load a pretrained YOLO model and run it on the bus image URL.
# Hint:
#   1. Create a YOLO object using the nano weights file: "yolo11n.pt"
#   2. Run the model on: "https://ultralytics.com/images/bus.jpg"
#   3. Display the first result.

model = ...

results = ...

# Display the first prediction result
...


## 1. Download a small set of example images

The images below are public sample images from the Ultralytics assets repository.  
You may replace them with your own images by changing the `IMAGE_URLS` dictionary.

In [ ]:
from pathlib import Path
import urllib.request

DATA_DIR = Path("data_yolo_practical")
DATA_DIR.mkdir(exist_ok=True)

IMAGE_URLS = {
    "bus.jpg": "https://ultralytics.com/images/bus.jpg",
    "zidane.jpg": "https://ultralytics.com/images/zidane.jpg",
}

image_paths = []

for filename, url in IMAGE_URLS.items():
    path = DATA_DIR / filename
    image_paths.append(path)

    if not path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, path)
    else:
        print(f"{filename} already exists.")

print(image_paths)

In [ ]:
def show_image_bgr(image_bgr, title=None, figsize=(8, 6)):
    """Display an OpenCV BGR image using matplotlib."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(image_rgb)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

img = cv2.imread(str(image_paths[0]))
show_image_bgr(img, title="Original image")

## 2. Load a pretrained YOLO model

Ultralytics automatically downloads the model weights the first time they are used.

`yolo11n.pt` is a small detection model. The `n` means *nano*. Larger variants, such as `s`, `m`, `l`, and `x`, are usually more accurate but slower and heavier.

In [ ]:
# TODO: Load the pretrained YOLO model.
# Keep MODEL_NAME as "yolo11n.pt" unless your tutor asks you to try a different model.

MODEL_NAME = "yolo11n.pt"

model = ...

model


## 3. Run object detection on one image

In [ ]:
# TODO: Run object detection on the first image.
# Use:
#   - source=str(single_image)
#   - conf=0.25
#   - imgsz=640
#   - verbose=False

single_image = image_paths[0]

results = model.predict(
    source=...,
    conf=...,
    imgsz=...,
    verbose=...
)

# TODO: Print/check the type of results and how many result objects were returned.
...


In [ ]:
# TODO: Select the first result and inspect its key attributes.
# Print:
#   - original image shape
#   - number of detections
#   - number of class names available

result = ...

print("Original image shape:", ...)
print("Number of detections:", ...)
print("Class names available:", ...)


## 4. Visualise detections

In [ ]:
# TODO: Visualise detections using result.plot().
# Remember: result.plot() returns an annotated image as a NumPy array in BGR order.

annotated = ...

show_image_bgr(..., title=f"Detections: {MODEL_NAME}")


## 5. Inspect bounding boxes, classes, and confidence scores

The `result.boxes` object contains the detection outputs. Important fields:

- `xyxy`: bounding boxes as `[x_min, y_min, x_max, y_max]`
- `conf`: confidence scores
- `cls`: predicted class indices
- `names`: mapping from class index to class name

In [ ]:
# TODO: Convert YOLO detections into a pandas DataFrame.
# Complete the steps below:
#   1. Extract bounding boxes from result.boxes.xyxy
#   2. Extract confidence scores from result.boxes.conf
#   3. Extract class IDs from result.boxes.cls
#   4. Build one dictionary per detection with:
#      class_id, class_name, confidence, x_min, y_min, x_max, y_max, width, height
#   5. Convert the list of dictionaries into a DataFrame.

boxes_xyxy = ...
confidences = ...
class_ids = ...

rows = []

for box, conf, cls_id in zip(boxes_xyxy, confidences, class_ids):
    x1, y1, x2, y2 = box

    rows.append({
        "class_id": ...,
        "class_name": ...,
        "confidence": ...,
        "x_min": ...,
        "y_min": ...,
        "x_max": ...,
        "y_max": ...,
        "width": ...,
        "height": ...,
    })

detections_df = ...

detections_df


### Exercise 1 — Concept check

**Question:** In this image, which object class has the highest confidence score?  


In [ ]:
# TODO: Answer Exercise 1 using code.
# Sort detections_df by confidence from highest to lowest and show the first row.

...


## 6. Filter detections by confidence

In practical applications, the confidence threshold strongly affects the final predictions.

- Lower threshold: more detections, but often more false positives.
- Higher threshold: fewer detections, but often fewer false positives and more missed objects.

In [ ]:
# TODO: Run inference with several confidence thresholds.
# For each threshold, print how many detections YOLO returns.

for threshold in [0.10, 0.25, 0.50, 0.75]:
    threshold_results = model.predict(
        source=...,
        conf=...,
        imgsz=...,
        verbose=...
    )

    threshold_result = ...
    print(...)


In [ ]:
# TODO: Visualise detections at a chosen confidence threshold.

threshold = 0.50

threshold_result = model.predict(
    source=...,
    conf=...,
    imgsz=...,
    verbose=...
)[0]

show_image_bgr(..., title=f"Detections with conf ≥ {threshold}")


### Exercise 2 — Threshold interpretation

**Question:** What is a possible downside of increasing the confidence threshold too much?  


## 7. Filter detections by class

YOLO models trained on COCO know 80 everyday object classes. We can filter detections after inference.

The example below keeps only detections predicted as `person`.

In [ ]:
# TODO: Filter the detection table to keep only one class.
# Start with "person", then try another class if available.

target_class_name = "person"

person_rows = ...

person_rows


In [ ]:
def draw_filtered_boxes(image_path, detection_df, class_name=None, min_conf=0.25):
    """Draw bounding boxes for selected detections.

    TODO:
    1. Read the image with OpenCV.
    2. Filter the DataFrame by confidence.
    3. Optionally filter by class name.
    4. Loop through the filtered rows.
    5. Draw each box and label using cv2.rectangle and cv2.putText.
    6. Return the image.
    """
    image = ...

    filtered = ...
    if class_name is not None:
        filtered = ...

    for _, row in filtered.iterrows():
        x1, y1, x2, y2 = ...
        label = ...

        # TODO: Draw rectangle and label.
        ...

    return image


filtered_img = draw_filtered_boxes(single_image, detections_df, class_name="person", min_conf=0.25)
show_image_bgr(filtered_img, title="Filtered detections: person only")


## 8. Batch inference on multiple images

In [ ]:
# TODO: Run batch inference on all downloaded images and summarise the results.
# Your summary should include:
#   - image filename
#   - number of detections
#   - sorted list of unique detected class names

batch_results = model.predict(
    source=...,
    conf=...,
    imgsz=...,
    verbose=...
)

summary_rows = []

for path, res in zip(image_paths, batch_results):
    class_ids = ...
    class_names = ...

    summary_rows.append({
        "image": ...,
        "num_detections": ...,
        "classes_detected": ...,
    })

batch_summary = ...

batch_summary


In [ ]:
# TODO: Visualise each batch prediction.

for path, res in zip(image_paths, batch_results):
    ...


## 9. Compare two model sizes

This section compares inference time and number of detections.  
For a short class, keep `MODEL_NAMES = ["yolo11n.pt", "yolo11s.pt"]`.  
If downloads are slow, use only `yolo11n.pt`.

In [ ]:
# TODO: Compare two YOLO model sizes.
# Measure inference time for each model and count detections.
# If downloads are slow, use only ["yolo11n.pt"].

MODEL_NAMES = ["yolo11n.pt", "yolo11s.pt"]

comparison_rows = []

for model_name in MODEL_NAMES:
    comparison_model = ...

    start = ...
    comparison_results = comparison_model.predict(
        source=...,
        conf=...,
        imgsz=...,
        verbose=...
    )
    elapsed = ...

    comparison_rows.append({
        "model": ...,
        "elapsed_seconds": ...,
        "num_detections": ...
    })

comparison_df = ...

comparison_df


### Exercise 3 — Model comparison

**Question:** Which model was faster? Did it detect the same number of objects?  


## 10. Optional: save detection outputs

In [ ]:
# TODO: Optional task — save detection outputs.
# Complete the prediction call so annotated predictions are saved to OUTPUT_DIR / "predictions".

OUTPUT_DIR = Path("outputs_yolo_practical")
OUTPUT_DIR.mkdir(exist_ok=True)

save_results = model.predict(
    source=...,
    conf=...,
    imgsz=...,
    save=...,
    project=...,
    name=...,
    exist_ok=...,
    verbose=...
)

print(f"Annotated predictions saved under: {OUTPUT_DIR / 'predictions'}")


## 11. Exercise 4 — Train a YOLO detector on a downloaded dataset

In this exercise, we move from **inference with pretrained weights** to **fine-tuning/training** on a small YOLO-format dataset.

We will remotely download the public `coco8` dataset from the Ultralytics assets repository. This dataset is intentionally tiny, so it is suitable for demonstrating the training workflow during a practical class. It is **not** large enough to produce a strong detector.

**Student task:** run the cells below, inspect the training outputs, and explain how the loss and metric curves change across epochs.


In [ ]:
from pathlib import Path
import zipfile
import urllib.request

# Data-loading code is provided.
# The public coco8 dataset is tiny and intended for demonstrating the YOLO training workflow.

TRAINING_DATA_DIR = Path("data_yolo_training")
TRAINING_DATA_DIR.mkdir(exist_ok=True)

DATASET_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip"
ZIP_PATH = TRAINING_DATA_DIR / "coco8.zip"
DATASET_DIR = TRAINING_DATA_DIR / "coco8"
DATASET_YAML = DATASET_DIR / "coco8.yaml"

if not DATASET_YAML.exists():
    print("Downloading coco8 dataset...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)

    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(TRAINING_DATA_DIR)
else:
    print("Dataset already exists.")

print("Dataset YAML:", DATASET_YAML)
print("Training images:", len(list((DATASET_DIR / "images" / "train").glob("*"))))
print("Validation images:", len(list((DATASET_DIR / "images" / "val").glob("*"))))


In [ ]:
# Inspect the dataset configuration file.
# YOLO dataset YAML files define the train/validation image folders and the class names.

print(DATASET_YAML.read_text())

In [ ]:
# TODO: Train/fine-tune a YOLO model.
# Keep the settings small for teaching purposes. Increase EPOCHS for a real experiment.
# Complete the code to:
#   1. Set training hyperparameters.
#   2. Load "yolo11n.pt".
#   3. Call .train(...) using DATASET_YAML.
#   4. Save the training run directory in RUN_DIR.

EPOCHS = ...
IMG_SIZE = ...
BATCH_SIZE = ...

train_model = ...

train_results = train_model.train(
    data=...,
    epochs=...,
    imgsz=...,
    batch=...,
    project=...,
    name=...,
    exist_ok=...,
    seed=...,
    verbose=...,
)

RUN_DIR = ...
print("Training run saved to:", RUN_DIR)


### Exercise 4a — Training configuration

In [ ]:
# TODO: Load the training log generated by Ultralytics.
# This CSV contains one row per epoch with losses and validation metrics.
# Strip leading/trailing spaces from the column names.

RESULTS_CSV = ...

metrics_df = ...

metrics_df.columns = ...

metrics_df.head()


In [ ]:
def plot_training_curves(metrics_df):
    """Plot YOLO training losses and validation metrics from results.csv.

    TODO:
    1. Identify the epoch column.
    2. Identify columns beginning with "train/" or "val/" as loss columns.
    3. Identify columns beginning with "metrics/" as metric columns.
    4. Plot losses.
    5. Plot validation metrics.
    """
    epoch_col = ...

    loss_cols = ...
    metric_cols = ...

    if loss_cols:
        plt.figure(figsize=(10, 6))
        for col in loss_cols:
            ...
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("YOLO training and validation losses")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    if metric_cols:
        plt.figure(figsize=(10, 6))
        for col in metric_cols:
            ...
        plt.xlabel("Epoch")
        plt.ylabel("Metric value")
        plt.title("YOLO validation metrics")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()


plot_training_curves(metrics_df)


In [ ]:
# TODO: Display the ready-made Ultralytics results plot if it exists.

results_plot = ...

if results_plot.exists():
    img = ...
    show_image_bgr(..., title="Ultralytics results.png", figsize=(12, 8))
else:
    print("results.png not found. Check that training completed successfully.")


In [ ]:
# TODO: Validate the best checkpoint and run inference on validation images.
# Complete the code to:
#   1. Load best.pt from RUN_DIR / "weights"
#   2. Validate the trained model
#   3. Predict on up to 4 validation images
#   4. Visualise each prediction

BEST_WEIGHTS = ...
trained_model = ...

validation_metrics = trained_model.val(data=..., imgsz=..., verbose=...)
print(validation_metrics)

val_images = ...
trained_predictions = trained_model.predict(
    source=...,
    conf=...,
    imgsz=...,
    verbose=...,
)

for path, pred in zip(val_images, trained_predictions):
    ...
